# Intro

This notebook is for running the LLM as a judge scoring for the OSQs. 

Framework:
- Pull in the OSQ response data
- (Optional) list of judge models -- default only Chatgpt5 or Chatgpt4o-mini
- List of Prompts
  - (Currently Omitted) Binary Correct/Incorrect
  - Scoring
    - Rubric 1 (from the MCQ -> OSQ conversion)
    - (Omitted) Rubric 2 (outside of recommended from conversion, e.g. from other sources)

# Configuration

In [ ]:
# ================================================
# Phase 5 — Append-Only Judging (Resumable, Single JSONL, Progress)
# ================================================
from pathlib import Path
from datetime import datetime
from statistics import mean
from tqdm.auto import tqdm
from openai import OpenAI
import os, json

# -----------------------------
# CONFIG
# -----------------------------
TASK_NAME    = "sysengbench-osq"
JUDGE_MODEL  = "openai/gpt-5"
# JUDGE_MODEL  = "openai/gpt-5-mini"
# JUDGE_MODEL = "google/gemini-2.5-flash"
TEMPERATURE  = 0.0
MAX_TOKENS   = 2000
SAMPLE_N     = 0          # 0 = judge ALL samples; else judge first N (for quick tests).
# If already judged > SAMPLE_N, it will error out. 
# TODO: Add logic to skip already-judged if SAMPLE_N is set and less than total.

# Paths (this notebook under: src/phase5_llm_as_a_judge/)
PHASE4_ROOT  = Path("../phase4_inference/output") / TASK_NAME
# PHASE5_ROOT  = Path(".") / TASK_NAME  # mirror structure in phase5
PHASE5_ROOT  = Path(".") / f"{TASK_NAME}-llm-judge" # Append "-llm-judge" to keep Phase 5 artifacts separate and consistent
# PHASE5_ROOT  = Path(".") / f"{TASK_NAME}-llm-judge-test" # Append "-llm-judge" to keep Phase 5 artifacts separate and consistent
PHASE5_ROOT.mkdir(parents=True, exist_ok=True)

# OpenRouter client
client = OpenAI(
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url="https://openrouter.ai/api/v1"
)
print("OPENROUTER_API_KEY present:", bool(os.getenv("OPENROUTER_API_KEY")))

OPENROUTER_API_KEY present: True


## Optional Troubleshooting Directory

In [17]:
# OPTIONAL TROUBLEHSOOTING CELL
from pathlib import Path
import json

p = Path("../phase4_inference/output/sysengbench-osq/gemma3__27b")  # or whichever model dir
f = sorted(p.glob("samples_*.jsonl"))[-1]  # newest
print(f"File = {f}")

with open(f,"r",encoding="utf-8") as fh:
    for i,line in enumerate(fh):
        if i>2: break
        print(json.loads(line))


File = ..\phase4_inference\output\sysengbench-osq\gemma3__27b\samples_sysengbench-osq_2025-11-15T01-24-12.251073.jsonl
{'doc_id': 0, 'doc': {'Question ID': 1, 'Tags': 'Introduction to risk', 'INCOSE Handbook Category': 'INCOSEHandbook/Systems Engineering Overview/System Concepts and Structures', 'question': 'What best describes the concept of uncertainty in systems engineering?', 'choiceA': 'The process of systematically improving and optimizing a system for efficiency.', 'choiceB': 'The condition where the outcomes of system functions are not predictable due to lack of information or variability.', 'choiceC': 'A method for analyzing the costs and benefits of a system over its lifecycle.', 'choiceD': 'The act of integrating different system components into a cohesive whole.', 'answer': 'B', 'label': 1, 'Justification': 'Uncertainty in systems engineering refers to the unpredictability of outcomes due to insufficient information or inherent variability within the system or its environme

# Check Number of Judgings to Perform

In [18]:
# this version takes into account multiple judges and multiple prompt numbers at the end (e.g -p1, -p2, etc)
import json
import pandas as pd
from pathlib import Path
import re

def build_multi_judge_progress_matrix(
    phase4_root: str,
    phase5_root: str,
    task_name: str
) -> pd.DataFrame:
    """
    Build a progress matrix for:
        (model, judge_name, prompt_number)

    File pattern expected:
        samples_<task>_<timestamp>__<judge_name>-p<number>.jsonl
    """

    p4 = Path(phase4_root)
    p5 = Path(phase5_root)

    if not p4.exists():
        raise FileNotFoundError(f"Phase 4 directory missing: {phase4_root}")

    # All models with Phase 4 output
    model_folders = sorted([d.name for d in (p4 / task_name).iterdir() if d.is_dir()])

    rows = []

    # Regex: Pattern matches __<judge_name>-p<number>.jsonl
    # Example: samples_sysengbench-osq_2025-11-16T21-02-49.453852__openai_gpt-5-p1.jsonl
    pattern = re.compile(r"__(.+?)-p(\d+)\.jsonl$")

    for model in model_folders:

        # Phase 4 source samples
        model_p4_dir = p4 / task_name / model

        # Phase 5 judged samples
        model_p5_dir = p5 / model

        # ------------------------------
        # TOTAL SAMPLES FROM PHASE 4
        # ------------------------------
        sample_files = [
            f for f in model_p4_dir.iterdir()
            if f.name.startswith(f"samples_{task_name}_") and f.suffix == ".jsonl"
        ]

        total_samples = 0
        if sample_files:
            sf = sample_files[0]
            with open(sf, "r", encoding="utf-8") as fh:
                total_samples = sum(1 for _ in fh)

        # ------------------------------
        # PER-JUDGE/PROMPT PROGRESS
        # ------------------------------

        judge_prompt_progress = {}  # (judge_name, pN) -> count

        if model_p5_dir.exists():
            for jf in model_p5_dir.iterdir():

                if not (jf.is_file() and jf.name.startswith("samples_") and jf.name.endswith(".jsonl")):
                    continue

                m = pattern.search(jf.name)
                if not m:
                    # Skip malformed names
                    continue

                judge_name = m.group(1)
                prompt_number = int(m.group(2))

                # Count judged samples
                with open(jf, "r", encoding="utf-8") as fh:
                    count = sum(1 for _ in fh)

                judge_prompt_progress[(judge_name, prompt_number)] = count

        # If no judged files found → record placeholder row
        if not judge_prompt_progress:
            judge_prompt_progress = {("no_judge_found", None): 0}

        # ----------------------------------------------
        # Build output rows: (model, judge_name, pN)
        # ----------------------------------------------
        for (judge_name, prompt_number), judged_count in judge_prompt_progress.items():

            if total_samples == 0:
                status = "not started"
            else:
                if judged_count == 0:
                    status = "not started"
                elif judged_count < total_samples:
                    status = "partial"
                else:
                    status = "complete"

            rows.append({
                "model_name": model,
                "ollama_name": model.replace("__", ":"),

                # NEW
                "judge_name": judge_name,
                "prompt_index": prompt_number,

                "judged_count": judged_count,
                "total_samples": total_samples,
                "progress_fraction": (
                    judged_count / total_samples if total_samples > 0 else 0.0
                ),
                "judge_status": status,
            })

    df = pd.DataFrame(rows)
    return df.sort_values(["model_name", "judge_name", "prompt_index"])


In [19]:

df = build_multi_judge_progress_matrix(
    phase4_root=str(PHASE4_ROOT.parent),
    phase5_root=str(PHASE5_ROOT),
    task_name=TASK_NAME
)

display(df)


,model_name,ollama_name,judge_name,prompt_index,judged_count,total_samples,progress_fraction,judge_status
0,anthropic__claude-sonnet-4.5,anthropic:claude-sonnet-4.5,google_gemini-2.5-flash,1,845,845,1.000000,complete
1,anthropic__claude-sonnet-4.5,anthropic:claude-sonnet-4.5,gpt-oss_120b,1,845,845,1.000000,complete
3,anthropic__claude-sonnet-4.5,anthropic:claude-sonnet-4.5,openai_gpt-5,1,845,845,1.000000,complete
2,anthropic__claude-sonnet-4.5,anthropic:claude-sonnet-4.5,openai_gpt-5-mini,1,845,845,1.000000,complete
5,devstral__24b,devstral:24b,openai_gpt-5,1,845,845,1.000000,complete
4,devstral__24b,devstral:24b,openai_gpt-5-mini,1,845,845,1.000000,complete
7,gemma3__12b,gemma3:12b,openai_gpt-5,1,845,845,1.000000,complete
6,gemma3__12b,gemma3:12b,openai_gpt-5-mini,1,845,845,1.000000,complete
9,gemma3__1b,gemma3:1b,openai_gpt-5,1,845,845,1.000000,complete
8,gemma3__1b,gemma3:1b,openai_gpt-5-mini,1,845,845,1.000000,complete


# LLM-as-a-Judge Inferencing

This section is broken up into Prompt > Sequential, Parallelized.

You can run one with a SAMPLE_N limit, then pickup where you left off with the other. The parallelized version is ~3-4x faster when running 8 workers (not 1:1 benefits...)

## Prompt 1: Only Scores

In [20]:
JUDGE_PROMPT = """You are an expert systems engineering (SE) educator evaluating student responses to open-ended questions.

EVALUATION SCHEMA (each 0–20, total 100):
1. TECHNICAL ACCURACY — correctness of SE concepts, terminology, and facts
2. CONCEPTUAL UNDERSTANDING — depth of comprehension of underlying SE principles
3. COMPLETENESS — coverage of key elements expected in the answer
4. CLARITY & ORGANIZATION — logical structure and clear communication
5. PROFESSIONAL RELEVANCE — connection to real-world SE practice and standards

QUESTION CONTEXT:
Original Question: {osq_question}
Expected Answer: {expected_answer}
Bloom's Level: {blooms_level}
SE Domain: {se_domain}

STUDENT RESPONSE TO EVALUATE:
{student_response}

INSTRUCTIONS:
Compare the response to the expected answer and criteria.
Return strict JSON in this format:
{{
  "technical_accuracy": {{"score": <0–20>}},
  "conceptual_understanding": {{"score": <0–20>}},
  "completeness": {{"score": <0–20>}},
  "clarity_organization": {{"score": <0–20>}},
  "professional_relevance": {{"score": <0–20>}},
  "overall_score": <0–100>
}}
"""


### Sequential

In [32]:
# ================================================
# Phase 5 — Append-Only Judging (Updated for new prompt & PROMPT_ID)
# ================================================
from pathlib import Path
from datetime import datetime
from tqdm import tqdm
from copy import deepcopy
import json
import os

# ------------------------------------------------
# CONFIG — Add this new line:
# ------------------------------------------------
PROMPT_ID = "p1"     # <--- CHANGE THIS TO "p2", "p3", etc. for different judging runs
# If PROMPT_ID = "", no suffix is added.
# ------------------------------------------------

# -----------------------------
# Helpers
# -----------------------------
def newest(path_iter):
    items = sorted([p for p in path_iter], key=lambda p: p.stat().st_mtime, reverse=True)
    return items[0] if items else None

def load_jsonl(path: Path):
    with open(path, "r", encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

def safe_json(s: str):
    try:
        return json.loads(s)
    except Exception:
        return None

def extract_student_response(sample_row):
    resps = sample_row.get("resps")
    if not resps:
        return ""
    first = resps[0]
    if isinstance(first, list) and first:
        first = first[0]
    return first if isinstance(first, str) else ""

def derive_prompt_fields(phase4_row):
    doc = (phase4_row.get("doc") or {}) if isinstance(phase4_row.get("doc"), dict) else {}
    osq_question    = doc.get("osq_prompt") or doc.get("question") or ""
    expected_answer = doc.get("expected_answer", "")
    blooms_level    = doc.get("blooms_level", "N/A")
    se_domain       = doc.get("INCOSE Handbook Category", "General Systems Engineering")
    student_resp    = extract_student_response(phase4_row)
    return {
        "osq_question": osq_question,
        "expected_answer": expected_answer,
        "student_response": student_resp,
        "blooms_level": blooms_level,
        "se_domain": se_domain,
    }

def triad_missing(fields):
    return [
        k for k in ("osq_question", "expected_answer", "student_response")
        if not isinstance(fields.get(k, ""), str) or not fields.get(k, "").strip()
    ]

def ensure_alignment_or_die(existing_row, current_src_row, sample_id):
    snap = existing_row.get("phase4_row")
    if snap is None:
        raise RuntimeError(
            f"[ALIGNMENT ERROR] sample_id={sample_id} has no 'phase4_row' snapshot in Phase-5 file."
        )
    if snap != current_src_row:
        raise RuntimeError(
            f"\n[ALIGNMENT ERROR] sample_id={sample_id} Phase-4 content changed.\n"
            f"Refusing to proceed. Freeze Phase-4 or regenerate Phase-5 from scratch."
        )


# -----------------------------
# Preconditions
# -----------------------------
if not PHASE4_ROOT.exists():
    raise FileNotFoundError(f"Phase-4 task directory not found: {PHASE4_ROOT}")

model_dirs = [d for d in PHASE4_ROOT.iterdir() if d.is_dir()]
if not model_dirs:
    raise FileNotFoundError(f"No model directories under {PHASE4_ROOT}")

print(f"Discovered {len(model_dirs)} model directories under task '{TASK_NAME}'.")

# -----------------------------
# Main loop per model
# -----------------------------
for model_dir in model_dirs:
    model_name = model_dir.name
    src_samples = newest(model_dir.glob("samples_*.jsonl"))
    if not src_samples:
        print(f"[skip] {model_name}: missing samples_*.jsonl")
        continue

    # Load Phase-4 source rows
    source_rows = load_jsonl(src_samples)
    if SAMPLE_N > 0:
        source_rows = source_rows[:SAMPLE_N]
    total = len(source_rows)
    print(f"\nModel: {model_name} | Source: {src_samples.name} | Count: {total}")

    out_dir = PHASE5_ROOT / model_name
    out_dir.mkdir(parents=True, exist_ok=True)

    phase4_name = src_samples.name.replace(".jsonl", "")
    judge_suffix = JUDGE_MODEL.replace("/", "_")

    # ------------------------------------------------
    # Filename with PROMPT_ID support
    # ------------------------------------------------
    if PROMPT_ID:
        target_name = f"{phase4_name}__{judge_suffix}-{PROMPT_ID}.jsonl"
    else:
        target_name = f"{phase4_name}__{judge_suffix}.jsonl"

    final_samples_path = out_dir / target_name

    # Resume behavior
    if final_samples_path.exists():
        print(f"[resume] Reusing existing samples file: {final_samples_path.name}")
    else:
        with open(final_samples_path, "w", encoding="utf-8"):
            pass
        print(f"[new] Creating samples file: {final_samples_path.name}")

    # Load existing judged rows
    done_ids = set()
    existing_by_id = {}
    existing_rows = load_jsonl(final_samples_path)

    for row in existing_rows:
        sid = row.get("sample_id")
        if isinstance(sid, int):
            done_ids.add(sid)
            existing_by_id[sid] = row

    # Alignment checks
    for sid in sorted(done_ids):
        if sid >= total:
            raise RuntimeError(
                f"{final_samples_path.name} has sample_id {sid} beyond current source length {total}."
            )
        ensure_alignment_or_die(existing_by_id[sid], source_rows[sid], sid)

    print(f"[progress] {model_name}: {len(done_ids)} already judged, {total - len(done_ids)} remaining.")

    # Determine which samples still need judging
    to_do = [i for i in range(total) if i not in done_ids]
    if not to_do:
        print(f"[done] {model_name}: nothing to judge.")
        continue

    with open(final_samples_path, "a", encoding="utf-8") as fout:
        for sid in tqdm(to_do, desc=f"Judging {model_name}", unit="sample"):
            phase4_row = deepcopy(source_rows[sid])
            fields_for_prompt = derive_prompt_fields(phase4_row)

            # Missing core fields → skipped record
            missing = triad_missing(fields_for_prompt)
            if missing:
                record = {
                    "sample_id": sid,
                    "phase4_row": phase4_row,
                    "judge": {
                        "fields": {
                            "technical_accuracy":      {"score": None},
                            "conceptual_understanding":{"score": None},
                            "completeness":            {"score": None},
                            "clarity_organization":    {"score": None},
                            "professional_relevance":  {"score": None},
                            "overall_score": None
                        },
                        "prompt": None,
                        "raw_output": None,
                        "timestamp": datetime.now().isoformat(),
                        "meta": {
                            "judge_model": JUDGE_MODEL,
                            "temperature": TEMPERATURE,
                            "max_tokens": MAX_TOKENS,
                            "task_name": TASK_NAME,
                            "model_name": model_name,
                            "prompt_id": PROMPT_ID,
                        }
                    }
                }
                fout.write(json.dumps(record) + "\n"); fout.flush()
                continue

            # Build judge prompt
            prompt = JUDGE_PROMPT.format(**fields_for_prompt)

            raw = None
            parsed = None
            api_error = None
            try:
                completion = client.chat.completions.create(
                    model=JUDGE_MODEL,
                    temperature=TEMPERATURE,
                    max_tokens=MAX_TOKENS,
                    messages=[
                        {"role": "system", "content": "You are an expert evaluator in systems engineering education."},
                        {"role": "user", "content": prompt}
                    ]
                )
                raw = (completion.choices[0].message.content or "").strip()
                parsed = safe_json(raw)
            except Exception as e:
                api_error = str(e)

            fields = {
                "technical_accuracy":      {"score": None},
                "conceptual_understanding":{"score": None},
                "completeness":            {"score": None},
                "clarity_organization":    {"score": None},
                "professional_relevance":  {"score": None},
                "overall_score": None
            }

            if parsed is not None:
                fields["technical_accuracy"]["score"]       = (parsed.get("technical_accuracy") or {}).get("score")
                fields["conceptual_understanding"]["score"] = (parsed.get("conceptual_understanding") or {}).get("score")
                fields["completeness"]["score"]             = (parsed.get("completeness") or {}).get("score")
                fields["clarity_organization"]["score"]     = (parsed.get("clarity_organization") or {}).get("score")
                fields["professional_relevance"]["score"]   = (parsed.get("professional_relevance") or {}).get("score")
                fields["overall_score"]                     = parsed.get("overall_score")

            record = {
                "sample_id": sid,
                "phase4_row": phase4_row,
                "judge": {
                    "fields": fields,
                    "prompt": prompt,
                    "raw_output": raw,
                    "timestamp": datetime.now().isoformat(),
                    "meta": {
                        "judge_model": JUDGE_MODEL,
                        "temperature": TEMPERATURE,
                        "max_tokens": MAX_TOKENS,
                        "task_name": TASK_NAME,
                        "model_name": model_name,
                        "prompt_id": PROMPT_ID,
                    }
                }
            }
            fout.write(json.dumps(record) + "\n")
            fout.flush()

    print(f"[✓] Samples appended for {model_name}: {final_samples_path.name}")

print("\nAll models processed.")


Discovered 8 model directories under task 'sysengbench-osq'.

Model: anthropic__claude-sonnet-4.5 | Source: samples_sysengbench-osq_2025-11-16T21-02-49.453852.jsonl | Count: 11
[resume] Reusing existing samples file: samples_sysengbench-osq_2025-11-16T21-02-49.453852__openai_gpt-5-p1.jsonl
[progress] anthropic__claude-sonnet-4.5: 10 already judged, 1 remaining.


Judging anthropic__claude-sonnet-4.5: 100%|██████████| 1/1 [00:16<00:00, 16.92s/sample]


[✓] Samples appended for anthropic__claude-sonnet-4.5: samples_sysengbench-osq_2025-11-16T21-02-49.453852__openai_gpt-5-p1.jsonl

Model: gemma3__27b | Source: samples_sysengbench-osq_2025-11-15T01-24-12.251073.jsonl | Count: 11
[resume] Reusing existing samples file: samples_sysengbench-osq_2025-11-15T01-24-12.251073__openai_gpt-5-p1.jsonl
[progress] gemma3__27b: 10 already judged, 1 remaining.


Judging gemma3__27b: 100%|██████████| 1/1 [00:21<00:00, 21.79s/sample]


[✓] Samples appended for gemma3__27b: samples_sysengbench-osq_2025-11-15T01-24-12.251073__openai_gpt-5-p1.jsonl

Model: google__gemini-2.5-flash | Source: samples_sysengbench-osq_2025-11-16T04-10-37.215910.jsonl | Count: 11
[resume] Reusing existing samples file: samples_sysengbench-osq_2025-11-16T04-10-37.215910__openai_gpt-5-p1.jsonl
[progress] google__gemini-2.5-flash: 10 already judged, 1 remaining.


...

[✓] Samples appended for mistral-large__123b: samples_sysengbench-osq_2025-11-15T00-00-12.395367__openai_gpt-5-p1.jsonl

Model: openai__gpt-4.1 | Source: samples_sysengbench-osq_2025-11-16T03-01-37.800397.jsonl | Count: 11
[resume] Reusing existing samples file: samples_sysengbench-osq_2025-11-16T03-01-37.800397__openai_gpt-5-p1.jsonl
[progress] openai__gpt-4.1: 10 already judged, 1 remaining.


Judging openai__gpt-4.1: 100%|██████████| 1/1 [00:20<00:00, 20.23s/sample]


[✓] Samples appended for openai__gpt-4.1: samples_sysengbench-osq_2025-11-16T03-01-37.800397__openai_gpt-5-p1.jsonl

Model: phi4__14b | Source: samples_sysengbench-osq_2025-11-14T23-10-44.863695.jsonl | Count: 11
[resume] Reusing existing samples file: samples_sysengbench-osq_2025-11-14T23-10-44.863695__openai_gpt-5-p1.jsonl
[progress] phi4__14b: 10 already judged, 1 remaining.


Judging phi4__14b: 100%|██████████| 1/1 [00:21<00:00, 21.75s/sample]

[✓] Samples appended for phi4__14b: samples_sysengbench-osq_2025-11-14T23-10-44.863695__openai_gpt-5-p1.jsonl

All models processed.


### Parallelized

Working with http requests since openai method doesnt allow multiple requests

TODO --- if results come back null (aka token/credit limits on OpenRouter), stop and throw an error message.

In [21]:
# ======================================================================
# Standalone Cell — Show EXACTLY what Phase-5 will judge for each model
# ======================================================================

from pathlib import Path
import json

def load_jsonl(path: Path):
    with open(path, "r", encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

def newest(path_iter):
    items = sorted([p for p in path_iter], key=lambda p: p.stat().st_mtime, reverse=True)
    return items[0] if items else None

print("=== Phase-5 Preview Report ===\n")
print(f"Task: {TASK_NAME}")
print(f"Judge Model: {JUDGE_MODEL}")
print(f"Prompt ID: {PROMPT_ID}\n")

model_dirs = [d for d in PHASE4_ROOT.iterdir() if d.is_dir()]
model_dirs = sorted(model_dirs, key=lambda x: x.name)

for model_dir in model_dirs:
    model_name = model_dir.name
    phase4_file = newest(model_dir.glob("samples_*.jsonl"))

    if not phase4_file:
        print(f"[skip] {model_name} — no Phase-4 sample file found.")
        continue

    phase4_rows = load_jsonl(phase4_file)
    total_samples = len(phase4_rows)

    # Derive expected Phase-5 output filename
    judge_suffix = JUDGE_MODEL.replace("/", "_")
    phase4_name = phase4_file.name.replace(".jsonl", "")

    if PROMPT_ID:
        phase5_filename = f"{phase4_name}__{judge_suffix}-{PROMPT_ID}.jsonl"
    else:
        phase5_filename = f"{phase4_name}__{judge_suffix}.jsonl"

    phase5_path = PHASE5_ROOT / model_name / phase5_filename

    if phase5_path.exists():
        existing_rows = load_jsonl(phase5_path)
        done_ids = {row.get("sample_id") for row in existing_rows if isinstance(row.get("sample_id"), int)}
        already_judged = len(done_ids)
    else:
        already_judged = 0

    remaining = total_samples - already_judged

    print(f"Model: {model_name}")
    print(f"  Phase-4 samples:     {total_samples}")
    print(f"  Already judged:      {already_judged}")
    print(f"  To judge now:        {remaining}")

    if remaining > 0:
        print(f"  → Phase-5 will judge sample IDs:")
        rem_ids = [i for i in range(total_samples) if i not in done_ids] if already_judged > 0 else list(range(total_samples))
        print(f"    {rem_ids[:20]}{' ...' if len(rem_ids) > 20 else ''}")

    print()
    
print("=== End of Preview ===")


=== Phase-5 Preview Report ===

Task: sysengbench-osq
Judge Model: openai/gpt-5
Prompt ID: p1

Model: anthropic__claude-sonnet-4.5
  Phase-4 samples:     845
  Already judged:      845
  To judge now:        0

Model: devstral__24b
  Phase-4 samples:     845
  Already judged:      845
  To judge now:        0

Model: gemma3__12b
  Phase-4 samples:     845
  Already judged:      845
  To judge now:        0

Model: gemma3__1b
  Phase-4 samples:     845
  Already judged:      845
  To judge now:        0

Model: gemma3__27b
  Phase-4 samples:     845
  Already judged:      845
  To judge now:        0

Model: gemma3__4b
  Phase-4 samples:     845
  Already judged:      845
  To judge now:        0

Model: google__gemini-2.5-flash
  Phase-4 samples:     845
  Already judged:      845
  To judge now:        0

Model: llama3.2__1b
  Phase-4 samples:     845
  Already judged:      845
  To judge now:        0

Model: llama3.2__3b
  Phase-4 samples:     845
  Already judged:      0
  To judge

In [ ]:
# ================================================================
# Phase 5 — Append-Only Judging (Parallel + OpenRouter-safe)
# ================================================================

import os
import json
import requests
from datetime import datetime
from pathlib import Path
from copy import deepcopy
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm


# ================================================================
# CONFIG — You MUST ensure these exist BEFORE this script runs
# (these already exist in your environment)
# ================================================================
PROMPT_ID = "p1"
MAX_WORKERS = os.cpu_count() or 4

# These must be defined exactly as in your existing pipeline:
# PHASE4_ROOT
# PHASE5_ROOT
# JUDGE_MODEL
# TEMPERATURE
# MAX_TOKENS
# TASK_NAME
# JUDGE_PROMPT
# SAMPLE_N
#
# If you want, I can generate a combined script with all config included.


# ================================================================
# Helper Functions (unchanged from your script)
# ================================================================
def newest(path_iter):
    items = sorted([p for p in path_iter], key=lambda p: p.stat().st_mtime, reverse=True)
    return items[0] if items else None

def load_jsonl(path: Path):
    with open(path, "r", encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

def safe_json(s: str):
    try:
        return json.loads(s)
    except:
        return None

def extract_student_response(sample_row):
    resps = sample_row.get("resps")
    if not resps:
        return ""
    first = resps[0]
    if isinstance(first, list) and first:
        first = first[0]
    return first if isinstance(first, str) else ""

def derive_prompt_fields(phase4_row):
    doc = (phase4_row.get("doc") or {}) if isinstance(phase4_row.get("doc"), dict) else {}
    osq_question    = doc.get("osq_prompt") or doc.get("question") or ""
    expected_answer = doc.get("expected_answer", "")
    blooms_level    = doc.get("blooms_level", "N/A")
    se_domain       = doc.get("INCOSE Handbook Category", "General Systems Engineering")
    student_resp    = extract_student_response(phase4_row)

    return {
        "osq_question": osq_question,
        "expected_answer": expected_answer,
        "student_response": student_resp,
        "blooms_level": blooms_level,
        "se_domain": se_domain,
    }

def triad_missing(fields):
    return [
        k for k in ("osq_question", "expected_answer", "student_response")
        if not isinstance(fields.get(k, ""), str) or not fields.get(k, "").strip()
    ]

def ensure_alignment_or_die(existing_row, current_src_row, sample_id):
    snap = existing_row.get("phase4_row")
    if snap is None:
        raise RuntimeError(
            f"[ALIGNMENT ERROR] sample_id={sample_id} missing phase4_row snapshot."
        )
    if snap != current_src_row:
        raise RuntimeError(
            f"[ALIGNMENT ERROR] sample_id={sample_id} Phase-4 changed.\n"
            "Regenerate Phase-5 from scratch."
        )


# ================================================================
# WORKER — SAFE WITH OPENROUTER
# Uses a fresh HTTP request per task
# ================================================================
def judge_worker(args):
    (
        sid,
        phase4_row,
        fields_for_prompt,
        JUDGE_PROMPT,
        JUDGE_MODEL,
        TEMPERATURE,
        MAX_TOKENS,
        TASK_NAME,
        model_name,
        PROMPT_ID,
    ) = args

    prompt = JUDGE_PROMPT.format(**fields_for_prompt)

    # Missing fields → skipped
    missing = [
        k for k in ("osq_question", "expected_answer", "student_response")
        if not fields_for_prompt.get(k, "").strip()
    ]
    if missing:
        return sid, {
            "sample_id": sid,
            "phase4_row": phase4_row,
            "judge": {
                "fields": None,
                "prompt": prompt,
                "raw_output": None,
                "error": f"missing_fields: {missing}",
                "timestamp": datetime.now().isoformat(),
                "meta": {
                    "judge_model": JUDGE_MODEL,
                    "temperature": TEMPERATURE,
                    "max_tokens": MAX_TOKENS,
                    "task_name": TASK_NAME,
                    "model_name": model_name,
                    "prompt_id": PROMPT_ID,
                }
            }
        }

    # ================================================================
    # OPENROUTER HTTP REQUEST (parallel-safe)
    # ================================================================
    try:
        response = requests.post(
            "https://openrouter.ai/api/v1/chat/completions",
            headers={
                "Authorization": f"Bearer {os.environ['OPENROUTER_API_KEY']}",
                "Content-Type": "application/json",
            },
            json={
                "model": JUDGE_MODEL,
                "messages": [
                    {"role": "system", "content": "You are an expert evaluator in systems engineering education."},
                    {"role": "user", "content": prompt},
                ],
                "temperature": TEMPERATURE,
                "max_tokens": MAX_TOKENS,
            },
            timeout=60,
        )

        data = response.json()

        raw = data["choices"][0]["message"]["content"]
        parsed = safe_json(raw)

    except Exception as e:
        return sid, {
            "sample_id": sid,
            "phase4_row": phase4_row,
            "judge": {
                "fields": None,
                "prompt": prompt,
                "raw_output": None,
                "error": str(e),
                "timestamp": datetime.now().isoformat(),
                "meta": {
                    "judge_model": JUDGE_MODEL,
                    "temperature": TEMPERATURE,
                    "max_tokens": MAX_TOKENS,
                    "task_name": TASK_NAME,
                    "model_name": model_name,
                    "prompt_id": PROMPT_ID,
                }
            }
        }

    # ================================================================
    # PARSE RESULT
    # ================================================================
    fields = {
        "technical_accuracy":      {"score": None},
        "conceptual_understanding":{"score": None},
        "completeness":            {"score": None},
        "clarity_organization":    {"score": None},
        "professional_relevance":  {"score": None},
        "overall_score": None
    }

    if parsed:
        fields["technical_accuracy"]["score"]       = (parsed.get("technical_accuracy") or {}).get("score")
        fields["conceptual_understanding"]["score"] = (parsed.get("conceptual_understanding") or {}).get("score")
        fields["completeness"]["score"]             = (parsed.get("completeness") or {}).get("score")
        fields["clarity_organization"]["score"]     = (parsed.get("clarity_organization") or {}).get("score")
        fields["professional_relevance"]["score"]   = (parsed.get("professional_relevance") or {}).get("score")
        fields["overall_score"]                     = parsed.get("overall_score")

    return sid, {
        "sample_id": sid,
        "phase4_row": phase4_row,
        "judge": {
            "fields": fields,
            "prompt": prompt,
            "raw_output": raw,
            "error": None,
            "timestamp": datetime.now().isoformat(),
            "meta": {
                "judge_model": JUDGE_MODEL,
                "temperature": TEMPERATURE,
                "max_tokens": MAX_TOKENS,
                "task_name": TASK_NAME,
                "model_name": model_name,
                "prompt_id": PROMPT_ID,
            }
        }
    }


# ================================================================
# MAIN EXECUTION
# ================================================================
def main():

    if not PHASE4_ROOT.exists():
        raise FileNotFoundError(f"Phase-4 directory not found: {PHASE4_ROOT}")

    model_dirs = [d for d in PHASE4_ROOT.iterdir() if d.is_dir()]
    print(f"Discovered {len(model_dirs)} model directories under task '{TASK_NAME}'.")

    for model_dir in model_dirs:
        model_name = model_dir.name
        src_samples = newest(model_dir.glob("samples_*.jsonl"))
        if not src_samples:
            print(f"[skip] {model_name}: no Phase-4 samples.")
            continue

        source_rows = load_jsonl(src_samples)
        if SAMPLE_N > 0:
            source_rows = source_rows[:SAMPLE_N]

        total = len(source_rows)
        print(f"\nModel: {model_name} | {src_samples.name} | Count: {total}")

        out_dir = PHASE5_ROOT / model_name
        out_dir.mkdir(parents=True, exist_ok=True)

        # File naming
        phase4_name = src_samples.name.replace(".jsonl", "")
        judge_suffix = JUDGE_MODEL.replace("/", "_")

        if PROMPT_ID:
            filename = f"{phase4_name}__{judge_suffix}-{PROMPT_ID}.jsonl"
        else:
            filename = f"{phase4_name}__{judge_suffix}.jsonl"

        final_path = out_dir / filename

        # Resume
        if not final_path.exists():
            with open(final_path, "w", encoding="utf-8"):
                pass
            print(f"[new] Created {filename}")
        else:
            print(f"[resume] Continuing {filename}")

        # Load previous rows
        done_ids = set()
        existing_by_id = {}

        for row in load_jsonl(final_path):
            sid = row.get("sample_id")
            if isinstance(sid, int):
                done_ids.add(sid)
                existing_by_id[sid] = row

        # Alignment
        for sid in done_ids:
            ensure_alignment_or_die(existing_by_id[sid], source_rows[sid], sid)

        print(f"[progress] {len(done_ids)} judged, {total - len(done_ids)} remaining.")

        # To-do list
        to_do = [i for i in range(total) if i not in done_ids]
        if not to_do:
            print(f"[done] {model_name}: nothing to judge.")
            continue

        # ================================================================
        # PARALLEL EXECUTION
        # ================================================================
        jobs = []
        for sid in to_do:
            phase4_row = deepcopy(source_rows[sid])
            fields = derive_prompt_fields(phase4_row)
            jobs.append((
                sid,
                phase4_row,
                fields,
                JUDGE_PROMPT,
                JUDGE_MODEL,
                TEMPERATURE,
                MAX_TOKENS,
                TASK_NAME,
                model_name,
                PROMPT_ID,
            ))

        print(f"[parallel] Dispatching {len(jobs)} tasks (workers={MAX_WORKERS})...")

        results = {}

        with ThreadPoolExecutor(max_workers=MAX_WORKERS) as pool:
            futures = {pool.submit(judge_worker, job): job[0] for job in jobs}

            for fut in tqdm(as_completed(futures), total=len(jobs),
                            desc=f"Judging {model_name}", unit="sample"):
                sid, row = fut.result()
                results[sid] = row

        # ================================================================
        # WRITE RESULTS
        # ================================================================
        with open(final_path, "a", encoding="utf-8") as fout:
            for sid in sorted(results.keys()):
                fout.write(json.dumps(results[sid]) + "\n")
                fout.flush()

        print(f"[✓] Completed: {filename}")

    print("\nAll models processed.")


# ================================================================
# ENTRYPOINT
# ================================================================
if __name__ == "__main__":
    main()


Discovered 19 model directories under task 'sysengbench-osq'.

Model: anthropic__claude-sonnet-4.5 | samples_sysengbench-osq_2025-11-16T21-02-49.453852.jsonl | Count: 845
[resume] Continuing samples_sysengbench-osq_2025-11-16T21-02-49.453852__openai_gpt-5-p1.jsonl
[progress] 845 judged, 0 remaining.
[done] anthropic__claude-sonnet-4.5: nothing to judge.

Model: devstral__24b | samples_sysengbench-osq_2025-11-19T22-36-33.281032.jsonl | Count: 845
[resume] Continuing samples_sysengbench-osq_2025-11-19T22-36-33.281032__openai_gpt-5-p1.jsonl
[progress] 845 judged, 0 remaining.
[done] devstral__24b: nothing to judge.

Model: gemma3__12b | samples_sysengbench-osq_2025-11-19T22-23-32.102126.jsonl | Count: 845
[resume] Continuing samples_sysengbench-osq_2025-11-19T22-23-32.102126__openai_gpt-5-p1.jsonl
[progress] 845 judged, 0 remaining.
[done] gemma3__12b: nothing to judge.

Model: gemma3__1b | samples_sysengbench-osq_2025-11-21T00-11-57.488573.jsonl | Count: 845
[resume] Continuing samples_s

Judging llama3.2__3b: 100%|██████████| 845/845 [24:41<00:00,  1.75s/sample]


[✓] Completed: samples_sysengbench-osq_2025-11-19T21-47-50.052256__openai_gpt-5-p1.jsonl

Model: llama3.3__70b | samples_sysengbench-osq_2025-11-14T23-24-28.915214.jsonl | Count: 845
[new] Created samples_sysengbench-osq_2025-11-14T23-24-28.915214__openai_gpt-5-p1.jsonl
[progress] 0 judged, 845 remaining.
[parallel] Dispatching 845 tasks (workers=8)...


Judging llama3.3__70b:  84%|████████▍ | 714/845 [21:41<04:19,  1.98s/sample]

### Testing a notification for when SAMPLES_N < already judged amount

In [ ]:
# ================================================================
# Phase 5 — Append-Only Judging (Parallel + OpenRouter-safe)
# ================================================================

import os
import json
import requests
from datetime import datetime
from pathlib import Path
from copy import deepcopy
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm


# ================================================================
# CONFIG — You MUST ensure these exist BEFORE this script runs
# (these already exist in your environment)
# ================================================================
PROMPT_ID = "p1"
MAX_WORKERS = os.cpu_count() or 4

# These must be defined exactly as in your existing pipeline:
# PHASE4_ROOT
# PHASE5_ROOT
# JUDGE_MODEL
# TEMPERATURE
# MAX_TOKENS
# TASK_NAME
# JUDGE_PROMPT
# SAMPLE_N
#
# If you want, I can generate a combined script with all config included.


# ================================================================
# Helper Functions (unchanged from your script)
# ================================================================
def newest(path_iter):
    items = sorted([p for p in path_iter], key=lambda p: p.stat().st_mtime, reverse=True)
    return items[0] if items else None

def load_jsonl(path: Path):
    with open(path, "r", encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

def safe_json(s: str):
    try:
        return json.loads(s)
    except:
        return None

def extract_student_response(sample_row):
    resps = sample_row.get("resps")
    if not resps:
        return ""
    first = resps[0]
    if isinstance(first, list) and first:
        first = first[0]
    return first if isinstance(first, str) else ""

def derive_prompt_fields(phase4_row):
    doc = (phase4_row.get("doc") or {}) if isinstance(phase4_row.get("doc"), dict) else {}
    osq_question    = doc.get("osq_prompt") or doc.get("question") or ""
    expected_answer = doc.get("expected_answer", "")
    blooms_level    = doc.get("blooms_level", "N/A")
    se_domain       = doc.get("INCOSE Handbook Category", "General Systems Engineering")
    student_resp    = extract_student_response(phase4_row)

    return {
        "osq_question": osq_question,
        "expected_answer": expected_answer,
        "student_response": student_resp,
        "blooms_level": blooms_level,
        "se_domain": se_domain,
    }

def triad_missing(fields):
    return [
        k for k in ("osq_question", "expected_answer", "student_response")
        if not isinstance(fields.get(k, ""), str) or not fields.get(k, "").strip()
    ]

def ensure_alignment_or_die(existing_row, current_src_row, sample_id):
    snap = existing_row.get("phase4_row")
    if snap is None:
        raise RuntimeError(
            f"[ALIGNMENT ERROR] sample_id={sample_id} missing phase4_row snapshot."
        )
    if snap != current_src_row:
        raise RuntimeError(
            f"[ALIGNMENT ERROR] sample_id={sample_id} Phase-4 changed.\n"
            "Regenerate Phase-5 from scratch."
        )


# ================================================================
# WORKER — SAFE WITH OPENROUTER
# Uses a fresh HTTP request per task
# ================================================================
def judge_worker(args):
    (
        sid,
        phase4_row,
        fields_for_prompt,
        JUDGE_PROMPT,
        JUDGE_MODEL,
        TEMPERATURE,
        MAX_TOKENS,
        TASK_NAME,
        model_name,
        PROMPT_ID,
    ) = args

    prompt = JUDGE_PROMPT.format(**fields_for_prompt)

    # Missing fields → skipped
    missing = [
        k for k in ("osq_question", "expected_answer", "student_response")
        if not fields_for_prompt.get(k, "").strip()
    ]
    if missing:
        return sid, {
            "sample_id": sid,
            "phase4_row": phase4_row,
            "judge": {
                "fields": None,
                "prompt": prompt,
                "raw_output": None,
                "error": f"missing_fields: {missing}",
                "timestamp": datetime.now().isoformat(),
                "meta": {
                    "judge_model": JUDGE_MODEL,
                    "temperature": TEMPERATURE,
                    "max_tokens": MAX_TOKENS,
                    "task_name": TASK_NAME,
                    "model_name": model_name,
                    "prompt_id": PROMPT_ID,
                }
            }
        }

    # ================================================================
    # OPENROUTER HTTP REQUEST (parallel-safe)
    # ================================================================
    try:
        response = requests.post(
            "https://openrouter.ai/api/v1/chat/completions",
            headers={
                "Authorization": f"Bearer {os.environ['OPENROUTER_API_KEY']}",
                "Content-Type": "application/json",
            },
            json={
                "model": JUDGE_MODEL,
                "messages": [
                    {"role": "system", "content": "You are an expert evaluator in systems engineering education."},
                    {"role": "user", "content": prompt},
                ],
                "temperature": TEMPERATURE,
                "max_tokens": MAX_TOKENS,
            },
            timeout=60,
        )

        data = response.json()

        raw = data["choices"][0]["message"]["content"]
        parsed = safe_json(raw)

    except Exception as e:
        return sid, {
            "sample_id": sid,
            "phase4_row": phase4_row,
            "judge": {
                "fields": None,
                "prompt": prompt,
                "raw_output": None,
                "error": str(e),
                "timestamp": datetime.now().isoformat(),
                "meta": {
                    "judge_model": JUDGE_MODEL,
                    "temperature": TEMPERATURE,
                    "max_tokens": MAX_TOKENS,
                    "task_name": TASK_NAME,
                    "model_name": model_name,
                    "prompt_id": PROMPT_ID,
                }
            }
        }

    # ================================================================
    # PARSE RESULT
    # ================================================================
    fields = {
        "technical_accuracy":      {"score": None},
        "conceptual_understanding":{"score": None},
        "completeness":            {"score": None},
        "clarity_organization":    {"score": None},
        "professional_relevance":  {"score": None},
        "overall_score": None
    }

    if parsed:
        fields["technical_accuracy"]["score"]       = (parsed.get("technical_accuracy") or {}).get("score")
        fields["conceptual_understanding"]["score"] = (parsed.get("conceptual_understanding") or {}).get("score")
        fields["completeness"]["score"]             = (parsed.get("completeness") or {}).get("score")
        fields["clarity_organization"]["score"]     = (parsed.get("clarity_organization") or {}).get("score")
        fields["professional_relevance"]["score"]   = (parsed.get("professional_relevance") or {}).get("score")
        fields["overall_score"]                     = parsed.get("overall_score")

    return sid, {
        "sample_id": sid,
        "phase4_row": phase4_row,
        "judge": {
            "fields": fields,
            "prompt": prompt,
            "raw_output": raw,
            "error": None,
            "timestamp": datetime.now().isoformat(),
            "meta": {
                "judge_model": JUDGE_MODEL,
                "temperature": TEMPERATURE,
                "max_tokens": MAX_TOKENS,
                "task_name": TASK_NAME,
                "model_name": model_name,
                "prompt_id": PROMPT_ID,
            }
        }
    }


# ================================================================
# MAIN EXECUTION
# ================================================================
def main():

    if not PHASE4_ROOT.exists():
        raise FileNotFoundError(f"Phase-4 directory not found: {PHASE4_ROOT}")

    model_dirs = [d for d in PHASE4_ROOT.iterdir() if d.is_dir()]
    print(f"Discovered {len(model_dirs)} model directories under task '{TASK_NAME}'.")

    for model_dir in model_dirs:
        model_name = model_dir.name
        src_samples = newest(model_dir.glob("samples_*.jsonl"))
        if not src_samples:
            print(f"[skip] {model_name}: no Phase-4 samples.")
            continue

        source_rows = load_jsonl(src_samples)
        if SAMPLE_N > 0:
            source_rows = source_rows[:SAMPLE_N]

        total = len(source_rows)
        print(f"\nModel: {model_name} | {src_samples.name} | Count: {total}")

        out_dir = PHASE5_ROOT / model_name
        out_dir.mkdir(parents=True, exist_ok=True)

        # File naming
        phase4_name = src_samples.name.replace(".jsonl", "")
        judge_suffix = JUDGE_MODEL.replace("/", "_")

        if PROMPT_ID:
            filename = f"{phase4_name}__{judge_suffix}-{PROMPT_ID}.jsonl"
        else:
            filename = f"{phase4_name}__{judge_suffix}.jsonl"

        final_path = out_dir / filename

        # Resume
        if not final_path.exists():
            with open(final_path, "w", encoding="utf-8"):
                pass
            print(f"[new] Created {filename}")
        else:
            print(f"[resume] Continuing {filename}")

        # Load previous rows
        done_ids = set()
        existing_by_id = {}

        for row in load_jsonl(final_path):
            sid = row.get("sample_id")
            if isinstance(sid, int):
                done_ids.add(sid)
                existing_by_id[sid] = row

        # ------------------------------------------------------------
        # Safe alignment with auto-skip for out-of-range judged rows
        # (occurs when SAMPLE_N < previous Phase-5 judged count)
        # ------------------------------------------------------------
        out_of_range_ids = []

        for sid in sorted(done_ids):
            if sid >= len(source_rows):
                # This judged row is valid historically, but not part of the current SAMPLE_N run
                out_of_range_ids.append(sid)
                continue  # Do not fail — just exclude from this run
            ensure_alignment_or_die(existing_by_id[sid], source_rows[sid], sid)

        if out_of_range_ids:
            print(
                f"[notice] {model_name}: {len(out_of_range_ids)} previously judged sample_ids "
                f"({min(out_of_range_ids)}–{max(out_of_range_ids)}) exceed the current "
                f"Phase-4 sample count ({len(source_rows)}). These are excluded from this run "
                f"because SAMPLE_N={SAMPLE_N} reduced the working set."
            )

        print(f"[progress] {len(done_ids)} judged, {total - len(done_ids)} remaining.")

        # ------------------------------------------------------------
        # If remaining == 0, then this run needs no additional judging
        # ------------------------------------------------------------
        if (total - len(done_ids)) <= 0:
            print(
                f"[done] {model_name}: No additional judging required for this run "
                f"(previous judged results already cover SAMPLE_N={SAMPLE_N})."
            )
            continue

        # ------------------------------------------------------------
        # To-do list for this run (only sample_ids < SAMPLE_N)
        # ------------------------------------------------------------
        to_do = [i for i in range(total) if i not in done_ids]

        # ================================================================
        # PARALLEL EXECUTION
        # ================================================================
        jobs = []
        for sid in to_do:
            phase4_row = deepcopy(source_rows[sid])
            fields = derive_prompt_fields(phase4_row)
            jobs.append((
                sid,
                phase4_row,
                fields,
                JUDGE_PROMPT,
                JUDGE_MODEL,
                TEMPERATURE,
                MAX_TOKENS,
                TASK_NAME,
                model_name,
                PROMPT_ID,
            ))

        print(f"[parallel] Dispatching {len(jobs)} tasks (workers={MAX_WORKERS})...")

        results = {}

        with ThreadPoolExecutor(max_workers=MAX_WORKERS) as pool:
            futures = {pool.submit(judge_worker, job): job[0] for job in jobs}

            for fut in tqdm(as_completed(futures), total=len(jobs),
                            desc=f"Judging {model_name}", unit="sample"):
                sid, row = fut.result()
                results[sid] = row

        # ================================================================
        # WRITE RESULTS
        # ================================================================
        with open(final_path, "a", encoding="utf-8") as fout:
            for sid in sorted(results.keys()):
                fout.write(json.dumps(results[sid]) + "\n")
                fout.flush()

        print(f"[✓] Completed: {filename}")

    print("\nAll models processed.")


# ================================================================
# ENTRYPOINT
# ================================================================
if __name__ == "__main__":
    main()


In [ ]:
# ------------------------------------------------------------
# Safe alignment with auto-skip for out-of-range judged rows
# (occurs when SAMPLE_N < previous Phase-5 judged count)
# ------------------------------------------------------------
out_of_range_ids = []

for sid in sorted(done_ids):
    if sid >= len(source_rows):
        # This judged row is valid historically, but not part of the current SAMPLE_N run
        out_of_range_ids.append(sid)
        continue  # Do not fail — just exclude from this run
    ensure_alignment_or_die(existing_by_id[sid], source_rows[sid], sid)

if out_of_range_ids:
    print(
        f"[notice] {model_name}: {len(out_of_range_ids)} previously judged sample_ids "
        f"({min(out_of_range_ids)}–{max(out_of_range_ids)}) exceed the current "
        f"Phase-4 sample count ({len(source_rows)}). These are excluded from this run "
        f"because SAMPLE_N={SAMPLE_N} reduced the working set."
    )

print(f"[progress] {len(done_ids)} judged, {total - len(done_ids)} remaining.")

# ------------------------------------------------------------
# If remaining == 0, then this run needs no additional judging
# ------------------------------------------------------------
if (total - len(done_ids)) <= 0:
    print(
        f"[done] {model_name}: No additional judging required for this run "
        f"(previous judged results already cover SAMPLE_N={SAMPLE_N})."
    )
    continue

# ------------------------------------------------------------
# To-do list for this run (only sample_ids < SAMPLE_N)
# ------------------------------------------------------------
to_do = [i for i in range(total) if i not in done_ids]


## Prompt 2: Scores and Justification (Original)

In [17]:
# -----------------------------
# Judge prompt (0–100 via five 0–20 dims) — braces escaped for .format()
# -----------------------------
JUDGE_PROMPT = """You are an expert systems engineering (SE) educator evaluating student responses to open-ended questions.

EVALUATION SCHEMA (each 0–20, total 100):
1. TECHNICAL ACCURACY — correctness of SE concepts, terminology, and facts
2. CONCEPTUAL UNDERSTANDING — depth of comprehension of underlying SE principles
3. COMPLETENESS — coverage of key elements expected in the answer
4. CLARITY & ORGANIZATION — logical structure and clear communication
5. PROFESSIONAL RELEVANCE — connection to real-world SE practice and standards

QUESTION CONTEXT:
Original Question: {osq_question}
Expected Answer: {expected_answer}
Bloom's Level: {blooms_level}
SE Domain: {se_domain}

STUDENT RESPONSE TO EVALUATE:
{student_response}

INSTRUCTIONS:
Compare the response to the expected answer and criteria.
Return strict JSON in this format:
{{
  "technical_accuracy": {{"score": <0–20>, "justification": "<1–2 sentences>"}},
  "conceptual_understanding": {{"score": <0–20>, "justification": "<1–2 sentences>"}},
  "completeness": {{"score": <0–20>, "justification": "<1–2 sentences>"}},
  "clarity_organization": {{"score": <0–20>, "justification": "<1–2 sentences>"}},
  "professional_relevance": {{"score": <0–20>, "justification": "<1–2 sentences>"}},
  "overall_score": <0–100>,
  "overall_assessment": "<summary>",
  "key_strengths": "<strengths>",
  "improvement_areas": "<areas for improvement>"
}}
"""

In [ ]:
# ================================================
# Phase 5 — Append-Only Judging (Preserve ALL Phase-4 Metadata)
# ================================================
from pathlib import Path
from datetime import datetime
from tqdm import tqdm
from copy import deepcopy
import json
import os

# -----------------------------
# Add this line (NEW)
# -----------------------------
PROMPT_ID = "p2"   # e.g., "p1", "p2", "v3", etc.

# -----------------------------
# Helpers
# -----------------------------
def newest(path_iter):
    items = sorted([p for p in path_iter], key=lambda p: p.stat().st_mtime, reverse=True)
    return items[0] if items else None

def load_jsonl(path: Path):
    with open(path, "r", encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

def safe_json(s: str):
    try:
        return json.loads(s)
    except Exception:
        return None

def extract_student_response(sample_row):
    resps = sample_row.get("resps")
    if not resps:
        return ""
    first = resps[0]
    if isinstance(first, list) and first:
        first = first[0]
    return first if isinstance(first, str) else ""

def derive_prompt_fields(phase4_row):
    doc = (phase4_row.get("doc") or {}) if isinstance(phase4_row.get("doc"), dict) else {}
    osq_question    = doc.get("osq_prompt") or doc.get("question") or ""
    expected_answer = doc.get("expected_answer", "")
    blooms_level    = doc.get("blooms_level", "N/A")
    se_domain       = doc.get("INCOSE Handbook Category", "General Systems Engineering")
    student_resp    = extract_student_response(phase4_row)
    return {
        "osq_question": osq_question,
        "expected_answer": expected_answer,
        "student_response": student_resp,
        "blooms_level": blooms_level,
        "se_domain": se_domain,
    }

def triad_missing(fields):
    return [
        k for k in ("osq_question", "expected_answer", "student_response")
        if not isinstance(fields.get(k, ""), str) or not fields.get(k, "").strip()
    ]

def ensure_alignment_or_die(existing_row, current_src_row, sample_id):
    snap = existing_row.get("phase4_row")
    if snap is None:
        raise RuntimeError(
            f"[ALIGNMENT ERROR] sample_id={sample_id} has no 'phase4_row' snapshot in Phase-5 file."
        )
    if snap != current_src_row:
        raise RuntimeError(
            f"\n[ALIGNMENT ERROR] sample_id={sample_id} Phase-4 content changed.\n"
            f"Refusing to proceed. Freeze Phase-4 or regenerate Phase-5 from scratch."
        )

# -----------------------------
# Preconditions
# -----------------------------
if not PHASE4_ROOT.exists():
    raise FileNotFoundError(f"Phase-4 task directory not found: {PHASE4_ROOT}")

model_dirs = [d for d in PHASE4_ROOT.iterdir() if d.is_dir()]
if not model_dirs:
    raise FileNotFoundError(f"No model directories under {PHASE4_ROOT}")

print(f"Discovered {len(model_dirs)} model directories under task '{TASK_NAME}'.")

# -----------------------------
# Main loop per model
# -----------------------------
for model_dir in model_dirs:
    model_name = model_dir.name
    src_samples = newest(model_dir.glob("samples_*.jsonl"))
    if not src_samples:
        print(f"[skip] {model_name}: missing samples_*.jsonl")
        continue

    source_rows = load_jsonl(src_samples)
    if SAMPLE_N > 0:
        source_rows = source_rows[:SAMPLE_N]
    total = len(source_rows)
    print(f"\nModel: {model_name} | Source: {src_samples.name} | Count: {total}")

    out_dir = PHASE5_ROOT / model_name
    out_dir.mkdir(parents=True, exist_ok=True)

    # -----------------------------
    # Filename with PROMPT_ID support (NEW BLOCK)
    # -----------------------------
    phase4_name = src_samples.name.replace(".jsonl", "")
    judge_suffix = JUDGE_MODEL.replace("/", "_")

    if PROMPT_ID:
        target_name = f"{phase4_name}__{judge_suffix}-{PROMPT_ID}.jsonl"
    else:
        target_name = f"{phase4_name}__{judge_suffix}.jsonl"

    final_samples_path = out_dir / target_name
    # -----------------------------

    if final_samples_path.exists():
        print(f"[resume] Reusing existing samples file: {final_samples_path.name}")
    else:
        with open(final_samples_path, "w", encoding="utf-8"):
            pass
        print(f"[new] Creating samples file: {final_samples_path.name}")

    done_ids = set()
    existing_by_id = {}
    existing_rows = load_jsonl(final_samples_path)

    for row in existing_rows:
        sid = row.get("sample_id")
        if isinstance(sid, int):
            done_ids.add(sid)
            existing_by_id[sid] = row

    for sid in sorted(done_ids):
        if sid >= total:
            raise RuntimeError(
                f"{final_samples_path.name} has sample_id {sid} beyond current source length {total}."
            )
        ensure_alignment_or_die(existing_by_id[sid], source_rows[sid], sid)

    print(f"[progress] {model_name}: {len(done_ids)} already judged, {total - len(done_ids)} remaining.")

    to_do = [i for i in range(total) if i not in done_ids]
    if not to_do:
        print(f"[done] {model_name}: nothing to judge.")
        continue

    with open(final_samples_path, "a", encoding="utf-8") as fout:
        for sid in tqdm(to_do, desc=f"Judging {model_name}", unit="sample"):
            phase4_row = deepcopy(source_rows[sid])
            fields_for_prompt = derive_prompt_fields(phase4_row)

            missing = triad_missing(fields_for_prompt)
            if missing:
                # unchanged SKIPPED record...
                record = {
                    "sample_id": sid,
                    "phase4_row": phase4_row,
                    "judge": {
                        "fields": {
                            "technical_accuracy":      {"score": None, "justification": None},
                            "conceptual_understanding":{"score": None, "justification": None},
                            "completeness":            {"score": None, "justification": None},
                            "clarity_organization":    {"score": None, "justification": None},
                            "professional_relevance":  {"score": None, "justification": None},
                            "overall_score": None,
                            "overall_assessment": f"SKIPPED: missing fields {missing}",
                            "key_strengths": None,
                            "improvement_areas": None,
                        },
                        "prompt": None,
                        "raw_output": None,
                        "timestamp": datetime.now().isoformat(),
                        "meta": {
                            "judge_model": JUDGE_MODEL,
                            "temperature": TEMPERATURE,
                            "max_tokens": MAX_TOKENS,
                            "task_name": TASK_NAME,
                            "model_name": model_name
                        }
                    }
                }
                fout.write(json.dumps(record) + "\n"); fout.flush()
                continue

            prompt = JUDGE_PROMPT.format(**fields_for_prompt)

            raw = None
            parsed = None
            api_error = None
            try:
                completion = client.chat.completions.create(
                    model=JUDGE_MODEL,
                    temperature=TEMPERATURE,
                    max_tokens=MAX_TOKENS,
                    messages=[
                        {"role": "system", "content": "You are an expert evaluator in systems engineering education."},
                        {"role": "user", "content": prompt}
                    ]
                )
                raw = (completion.choices[0].message.content or "").strip()
                parsed = safe_json(raw)
            except Exception as e:
                api_error = str(e)

            # unchanged construction of fields...
            fields = {
                "technical_accuracy":      {"score": None, "justification": None},
                "conceptual_understanding":{"score": None, "justification": None},
                "completeness":            {"score": None, "justification": None},
                "clarity_organization":    {"score": None, "justification": None},
                "professional_relevance":  {"score": None, "justification": None},
                "overall_score": None,
                "overall_assessment": None,
                "key_strengths": None,
                "improvement_areas": None,
            }

            if parsed is None:
                fields["overall_assessment"] = "ERROR: Invalid JSON response" + (f" ({api_error})" if api_error else "")
            else:
                def sget(d, key):
                    v = d.get(key)
                    return (v or {}).get("score") if isinstance(v, dict) else v
                fields["technical_accuracy"]       = {"score": sget(parsed,"technical_accuracy"),      "justification": (parsed.get("technical_accuracy") or {}).get("justification")}
                fields["conceptual_understanding"] = {"score": sget(parsed,"conceptual_understanding"),"justification": (parsed.get("conceptual_understanding") or {}).get("justification")}
                fields["completeness"]             = {"score": sget(parsed,"completeness"),            "justification": (parsed.get("completeness") or {}).get("justification")}
                fields["clarity_organization"]     = {"score": sget(parsed,"clarity_organization"),    "justification": (parsed.get("clarity_organization") or {}).get("justification")}
                fields["professional_relevance"]   = {"score": sget(parsed,"professional_relevance"),  "justification": (parsed.get("professional_relevance") or {}).get("justification")}
                fields["overall_score"]            = parsed.get("overall_score")
                fields["overall_assessment"]       = parsed.get("overall_assessment")
                fields["key_strengths"]            = parsed.get("key_strengths")
                fields["improvement_areas"]        = parsed.get("improvement_areas")

            record = {
                "sample_id": sid,
                "phase4_row": phase4_row,
                "judge": {
                    "fields": fields,
                    "prompt": prompt,
                    "raw_output": raw,
                    "timestamp": datetime.now().isoformat(),
                    "meta": {
                        "judge_model": JUDGE_MODEL,
                        "temperature": TEMPERATURE,
                        "max_tokens": MAX_TOKENS,
                        "task_name": TASK_NAME,
                        "model_name": model_name
                    }
                }
            }
            fout.write(json.dumps(record) + "\n")
            fout.flush()

    print(f"[✓] Samples appended for {model_name}: {final_samples_path.name}")

print("\nAll models processed.")


### TODO: Parallelized of Prompt 2

# Cleanup Excess Judging Lines

**BE CAREFUL**: This is standalone and optional — run manually when you want to prune stale Phase-5 data.

In [ ]:
# ================================================================
# Cleanup Cell — Remove stale judged rows in Phase-5
# ================================================================

from pathlib import Path
import json

def load_jsonl(path):
    with open(path, "r", encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

def write_jsonl(path, rows):
    with open(path, "w", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r) + "\n")

print("=== Phase-5 Cleanup — Removing Stale Rows ===")
print(f"Task: {TASK_NAME}\n")

model_dirs = [d for d in PHASE4_ROOT.iterdir() if d.is_dir()]

for model_dir in model_dirs:
    model_name = model_dir.name
    phase4_file = sorted(model_dir.glob("samples_*.jsonl"))
    if not phase4_file:
        continue
    phase4_rows = load_jsonl(phase4_file[-1])
    n_phase4 = len(phase4_rows)

    # Find Phase-5 file
    p5_dir = PHASE5_ROOT / model_name
    if not p5_dir.exists():
        continue
    p5_files = list(p5_dir.glob("*.jsonl"))
    if not p5_files:
        continue

    for p5 in p5_files:
        rows = load_jsonl(p5)
        before = len(rows)
        rows = [r for r in rows if isinstance(r.get("sample_id"), int) and r["sample_id"] < n_phase4]
        after = len(rows)

        if after < before:
            print(f"{model_name}: removed {before - after} stale rows from {p5.name}")
            write_jsonl(p5, rows)

print("\nCleanup complete.")


# Note that no `results_` file is made. This is INTENTIONAL. Process everything in Phase 6 for analysis.
The results file would only include final accuracy. We can process that prior to analysis. 